# Najafi et al. (2020) on DANDI:000016, notebook 2: choice decoders 0-97 ms before the choice

**Paper.** Najafi F, Elsayed GF, Cao R, Pnevmatikakis E, Latham PE, Cunningham JP, Churchland AK (2020).
Excitatory and inhibitory subnetworks are equally selective during decision-making and emerge
simultaneously during learning. *Neuron* 105, 165-179. [doi:10.1016/j.neuron.2019.09.045](https://doi.org/10.1016/j.neuron.2019.09.045)

**Dataset.** [DANDI:000016](https://dandiarchive.org/dandiset/000016/0.260915.2309), version
0.260915.2309 (doi:10.48324/dandi.000016/0.260915.2309), streamed from the archive. This is the only
published version (2026-09-15); the paper cites no DANDI version (it names the NWB deposit at the
CSHL repository, doi:10.14224/1.37693, from which the dandiset is derived), and the draft holds
the same 135 files.

**Claims.** `claims/000016/10.1016-j.neuron.2019.09.045.yaml`, written from the accepted author
manuscript in PubMed Central (PMC6952547) with its supplement, and the paper's code
(github.com/farznaj/imaging_decisionMaking_exc_inh at commit b7352acd). The claims file decides what
is tested and how; this notebook does not change a claim.

Claims tested here (the full table is in notebook 1):

| Claim | Finding | Notebook | Status |
|---|---|---|---|
| `fig3bc-decoding-similar` | E and I populations decode choice equally well | 02 | tested |
| `fig3df-weights-similar` | Matched decoder weights of E and I | 02 (bin before choice), 03 (every bin) | tested |
| `figs6de-incorrect-trials` | Decoders generalise to incorrect trials similarly for E and I | 02 | tested |
| `fig4bd-stability-similar` | Similar stability of E and I decoders | 02 | tested |

Decoders follow the claims file's opening comment: correct trials, each neuron z-scored over them,
left and right choices matched by subsampling, 50 random 90/10 splits, C chosen from 61 values of
10^(-6 to 6)/number of trials by the smallest mean test error, and 50 random subsamples of as many
excitatory neurons as the session has inhibitory neurons. The SVM is solved exactly (najafi.py);
the data check compares it with scikit-learn's LinearSVC.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

import najafi as nj
import sessions as S

pd.set_option("display.width", 200)
MICE = [1, 2, 3, 4]
A = nj.asset_urls()
print(f"DANDI:{nj.DANDISET_ID} version {nj.DANDISET_VERSION}: {len(A)} NWB files")
items = [(i, p, u) for i, (p, u) in enumerate(zip(A.path, A.url))]
R = nj.map_sessions(S.decode_bin, items, workers=4)
OK = [r for r in R if r["ok"]]
period = np.mean([np.diff(r["bin_times"]).mean() for r in R]) / 3 / 1000   # frame period, s
print(len(R), "sessions loaded,", len(OK), "with at least 10 correct trials of each choice and 2 inhibitory neurons")

## Data check

In [ ]:
d = pd.DataFrame([dict(mouse=r["mouse"], ok=r["ok"], left=r["n_left"], right=r["n_right"], exc=r["n_exc"], inh=r["n_inh"],
                       all=r["n_all"], inc=sum(r.get("n_incorrect", (0, 0)))) for r in R])
print(d.groupby("mouse").agg(sessions=("ok", "size"), used=("ok", "sum"), correct_high_mean=("left", "mean"),
                             correct_low_mean=("right", "mean"), neurons_mean=("all", "mean"), inh_mean=("inh", "mean"),
                             exc_mean=("exc", "mean"), incorrect_mean=("inc", "mean")).round(1).to_string())
print("Sessions left out:", [(r["mouse"], r["date"], r["n_left"], r["n_right"], r["n_inh"]) for r in R if not r["ok"]])
print("Bin before the choice starts at %.1f ms (frame period %.2f ms)" % (OK[0]["bin_times"][OK[0]["i1"]], period * 1000))
ci = pd.Series([r["all"]["C_index"] for r in OK] + [r["inh"]["C_index"] for r in OK])
print("Index of the chosen C in the grid of 61 (all-neuron and inhibitory decoders): min %d, median %d, max %d"
      % (ci.min(), ci.median(), ci.max()))
print("Test trials per split: %d to %d; training trials %d to %d" % (min(r["all"]["n_test"] for r in OK), max(r["all"]["n_test"] for r in OK),
      min(r["all"]["n_train"] for r in OK), max(r["all"]["n_train"] for r in OK)))
# solver check against scikit-learn's LinearSVC (default settings) on one session
from sklearn.svm import LinearSVC
from sklearn.exceptions import ConvergenceWarning
r0 = OK[0]
s0 = nj.load_session(A.path[r0["idx"]], A.url[r0["idx"]], groups=("choice",))
xb, i1, _, corr, _ = S._choice_data(s0, "correct")
X = xb[corr][:, i1]; y = s0["trials"].choice_high.values[corr].astype(int)
for name, cols in (("all", np.arange(X.shape[1])), ("inh", np.where(s0["types"] == "inhibitory")[0])):
    rng = np.random.default_rng(0)
    sp = nj.splits(y, rng, 10)
    ours = nj.decode(X[:, cols], y, rng, sp=sp)
    Xz, _, _ = nj.zscore(X[:, cols])
    with warnings.catch_warnings(record=True) as wlist:
        warnings.simplefilter("always", ConvergenceWarning)
        acc = [100 * (LinearSVC(C=ours["C"]).fit(Xz[a], y[a]).predict(Xz[b]) == y[b]).mean() for a, b in sp]
    print(f"{name}: chosen C {ours['C']:.3g}; accuracy over 10 splits, this solver %.1f%%, LinearSVC %.1f%%; "
          f"LinearSVC convergence warnings: {len(wlist)}" % (ours["acc_reps"].mean(), np.mean(acc)))

## fig3bc-decoding-similar

> Choice can be decoded from population activity well above chance, and inhibitory populations
> decode it as accurately as excitatory populations of the same size, in every mouse.

Criterion: per session the accuracy at 0-97 ms before the choice, averaged over the 50 repetitions
(and the 50 excitatory subsamples). Per mouse, above chance when accuracy exceeds the shuffled
control by an unpaired two-tailed t-test across sessions at p < 0.05; equality, a claim of absence,
by the unpaired two-tailed t-test across sessions at p < 0.05. Reproduced when in every mouse both
types are above chance and do not differ; partially reproduced when they differ in one mouse.

In [ ]:
rows = []
for m in MICE:
    rr = [r for r in OK if r["mouse"] == m]
    a_all = np.array([r["all"]["acc"] for r in rr]); s_all = np.array([r["all"]["shfl"] for r in rr])
    a_inh = np.array([r["inh"]["acc"] for r in rr]); s_inh = np.array([r["inh"]["shfl"] for r in rr])
    a_exc = np.array([np.mean([e["acc"] for e in r["exc"]]) for r in rr])
    s_exc = np.array([np.mean([e["shfl"] for e in r["exc"]]) for r in rr])
    rows.append(dict(mouse=m, sessions=len(rr), all=a_all.mean(), all_shfl=s_all.mean(), p_all=nj.ttest(a_all, s_all)[1],
                     exc=a_exc.mean(), exc_shfl=s_exc.mean(), p_exc_vs_shfl=nj.ttest(a_exc, s_exc)[1],
                     inh=a_inh.mean(), inh_shfl=s_inh.mean(), p_inh_vs_shfl=nj.ttest(a_inh, s_inh)[1],
                     inh_minus_exc=a_inh.mean() - a_exc.mean(), p_exc_vs_inh=nj.ttest(a_exc, a_inh)[1]))
T3 = pd.DataFrame(rows)
print(T3.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
above = ((T3.exc > T3.exc_shfl) & (T3.p_exc_vs_shfl < 0.05) & (T3.inh > T3.inh_shfl) & (T3.p_inh_vs_shfl < 0.05))
print("Both types above the shuffle in every mouse:", bool(above.all()), "; mice where E and I differ:", T3.mouse[T3.p_exc_vs_inh < 0.05].tolist())
print("Accuracy ranges over sessions: all %.1f-%.1f, inh %.1f-%.1f, exc %.1f-%.1f (percent)" % (
    min(r["all"]["acc"] for r in OK), max(r["all"]["acc"] for r in OK), min(r["inh"]["acc"] for r in OK),
    max(r["inh"]["acc"] for r in OK), min(np.mean([e["acc"] for e in r["exc"]]) for r in OK),
    max(np.mean([e["acc"] for e in r["exc"]]) for r in OK)))

**Result: (to be written)**

## fig3df-weights-similar

> In the decoder trained on all neurons, excitatory and inhibitory neurons receive weights of
> matched magnitude throughout the trial and with very similar distributions.

Criterion: all-neuron decoder at the best C, weights averaged over the 50 repetitions, each
session's weight vector normalised to unit length; mean |w| per type per session at 0-97 ms before
the choice (per bin in notebook 3). Unpaired two-tailed t-test across sessions per mouse at
p < 0.05: reproduced when no mouse differs, partially reproduced when one or two do, not reproduced
otherwise. Pooled neuron counts compared with 42,019 excitatory and 5,172 inhibitory.

In [ ]:
rows = []
we_all, wi_all = [], []
for m in MICE:
    rr = [r for r in OK if r["mouse"] == m]
    me, mi = [], []
    for r in rr:
        w = r["all"]["w"] / np.linalg.norm(r["all"]["w"])
        t_ = r["all"]["w_types"]
        me.append(np.abs(w[t_ == "excitatory"]).mean()); mi.append(np.abs(w[t_ == "inhibitory"]).mean())
        we_all.append(w[t_ == "excitatory"]); wi_all.append(w[t_ == "inhibitory"])
    tt = nj.ttest(me, mi)
    rows.append(dict(mouse=m, sessions=len(rr), abs_w_exc=np.mean(me), abs_w_inh=np.mean(mi), t=tt[0], p=tt[1]))
T = pd.DataFrame(rows)
print(T.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
we_all, wi_all = np.concatenate(we_all), np.concatenate(wi_all)
print("Mice with p < 0.05:", T.mouse[T.p < 0.05].tolist())
print("Pooled neurons: excitatory %d (expected 42019 +/- 4202), inhibitory %d (expected 5172 +/- 517)" % (len(we_all), len(wi_all)))
print("Pooled weight distributions: excitatory mean %.4f, SD %.4f; inhibitory mean %.4f, SD %.4f; KS p = %.3g"
      % (we_all.mean(), we_all.std(), wi_all.mean(), wi_all.std(), stats.ks_2samp(we_all, wi_all).pvalue))

**Result: (to be written)**

## figs6de-incorrect-trials

> Decoders trained on correct trials predict the choice on incorrect trials with high accuracy,
> slightly lower than on correct trials, and the reduction is similar for excitatory and
> inhibitory populations in every mouse.

Criterion: decoders trained on correct trials at 0-97 ms before the choice, tested on held-out
correct trials and on incorrect trials; accuracy on incorrect trials is high when above the shuffle
by the unpaired two-tailed t-test across sessions per mouse; the reductions (correct minus
incorrect) of the inhibitory and subsampled excitatory decoders compared by the same test, a claim
of absence; mouse 4 should have the largest reduction for the all-neuron decoder. The incorrect
trials are scored whole by each repetition's decoder; their shuffle permutes their labels. How the
incorrect trials are normalised is not given; both readings are run: (i) with the correct trials'
mean and SD (the trained decoder's normalisation) and (ii) z-scored over the incorrect trials
themselves. Sessions need at least one incorrect trial of each choice.

In [ ]:
for norm in ("inc_train_norm", "inc_own_norm"):
    rows = []
    for m in MICE:
        rr = [r for r in OK if r["mouse"] == m and norm in r["all"] and min(r["n_incorrect"]) >= 1]
        ai = {k: np.array([r[k][norm].mean() for r in rr]) for k in ("all", "inh")}
        ai["exc"] = np.array([np.mean([e[norm].mean() for e in r["exc"]]) for r in rr])
        si = {k: np.array([r[k][norm + "_shfl"].mean() for r in rr]) for k in ("all", "inh")}
        si["exc"] = np.array([np.mean([e[norm + "_shfl"].mean() for e in r["exc"]]) for r in rr])
        ac = {k: np.array([r[k]["acc"] for r in rr]) for k in ("all", "inh")}
        ac["exc"] = np.array([np.mean([e["acc"] for e in r["exc"]]) for r in rr])
        red = {k: ac[k] - ai[k] for k in ac}
        row = dict(mouse=m, sessions=len(rr))
        for k in ("all", "exc", "inh"):
            row[f"{k}_correct"] = ac[k].mean(); row[f"{k}_incorrect"] = ai[k].mean(); row[f"{k}_inc_shfl"] = si[k].mean()
            row[f"p_{k}_inc_vs_shfl"] = nj.ttest(ai[k], si[k])[1]
        row["red_exc"], row["red_inh"], row["red_all"] = red["exc"].mean(), red["inh"].mean(), red["all"].mean()
        row["p_red_exc_vs_inh"] = nj.ttest(red["exc"], red["inh"])[1]
        rows.append(row)
    T = pd.DataFrame(rows)
    print(f"--- reading {norm}")
    print(T.T.to_string(float_format=lambda v: f"{v:.4g}"))
    above = all((T[f"{k}_incorrect"] > T[f"{k}_inc_shfl"]).all() and (T[f"p_{k}_inc_vs_shfl"] < 0.05).all() for k in ("all", "exc", "inh"))
    print("incorrect-trial accuracy above shuffle in every mouse (all, exc, inh):", above,
          "; mice where reductions differ:", T.mouse[T.p_red_exc_vs_inh < 0.05].tolist(),
          "; mouse with largest all-neuron reduction:", int(T.mouse[T.red_all.idxmax()]))

**Result: (to be written)**

## fig4bd-stability-similar

> Choice decoders lose accuracy at a similar rate away from their training time for excitatory
> and inhibitory populations, and their stability duration is indistinguishable.

Criterion: the decoder trained at 0-97 ms before the choice is tested at every bin on its held-out
trials (each bin z-scored over the correct trials); a test bin is stable when its accuracy is at
least the training-bin accuracy minus 2 SD across the 50 repetitions (Methods; 2 SE of the code as a
variant). Duration = stable bins x 97 ms (bins need not be contiguous; contiguous bins around the
training bin as a variant). Excitatory durations averaged over the 50 subsamples. Sessions whose
training-bin accuracy does not exceed the shuffle are left out (one-tailed t-test across
repetitions; alpha 0.001 of the code and 0.05 of the Methods, both readings; a session is left out
if its inhibitory decoder or its pooled excitatory decoders fail). Absence judged on the code's
unpaired two-tailed t-test of the four mouse means; the per-mouse test across sessions is a variant.

In [ ]:
binw = 3 * period * 1000
def durations(stab, i1, rule):
    acc = stab.mean(0); sd = stab.std(0, ddof=1)
    thr = acc[i1] - 2 * (sd[i1] if rule == "sd" else sd[i1] / np.sqrt(stab.shape[0]))
    st = acc >= thr
    lo = i1
    while lo - 1 >= 0 and st[lo - 1]: lo -= 1
    hi = i1
    while hi + 1 < len(st) and st[hi + 1]: hi += 1
    return st.sum() * binw, (hi - lo + 1) * binw
def exceeds(acc, sh, alpha):
    t, p = stats.ttest_ind(acc, sh)
    return t > 0 and p / 2 < alpha
out = []
for r in OK:
    i1 = r["i1"]
    for alpha in (0.001, 0.05):
        keep = exceeds(r["inh"]["acc_reps"], r["inh"]["shfl_reps"], alpha) and exceeds(
            np.concatenate([e["acc_reps"] for e in r["exc"]]), np.concatenate([e["shfl_reps"] for e in r["exc"]]), alpha)
        for rule in ("sd", "se"):
            di = durations(r["inh"]["stab"], i1, rule)
            de = np.mean([durations(e["stab"], i1, rule) for e in r["exc"]], 0)
            out.append(dict(mouse=r["mouse"], alpha=alpha, rule=rule, keep=keep, inh=di[0], exc=de[0], inh_contig=di[1], exc_contig=de[1]))
D = pd.DataFrame(out)
rows = []
for (alpha, rule), g in D[D.keep].groupby(["alpha", "rule"]):
    mm = g.groupby("mouse")[["exc", "inh", "exc_contig", "inh_contig"]].mean()
    per = {m: nj.ttest(g[g.mouse == m].exc, g[g.mouse == m].inh)[1] for m in MICE}
    rows.append(dict(alpha=alpha, rule=rule, sessions=len(g), mice=len(mm),
                     exc_means=np.round(mm.exc.values, 1).tolist(), inh_means=np.round(mm.inh.values, 1).tolist(),
                     p_mouse_means=nj.ttest(mm.exc, mm.inh)[1], p_mouse_means_contig=nj.ttest(mm.exc_contig, mm.inh_contig)[1],
                     per_mouse_p=[round(per[m], 3) for m in MICE]))
T = pd.DataFrame(rows)
print(T.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("Sessions kept per reading:", D.groupby(["alpha", "rule"]).keep.sum().to_dict())
# decay of accuracy away from the training bin (Figure 4B), mean over sessions per mouse
for m in MICE:
    rr = [r for r in OK if r["mouse"] == m]
    if not rr:
        continue
    ai, off = nj.align_bins([r["inh"]["stab"].mean(0) for r in rr], [r["i1"] for r in rr])
    ae, _ = nj.align_bins([np.mean([e["stab"].mean(0) for e in r["exc"]], 0) for r in rr], [r["i1"] for r in rr])
    print(f"mouse {m}: accuracy of the decoder trained at -1, tested at offsets {off[0]}..{off[-1]}:")
    print("   inh", np.round(ai.mean(0), 1).tolist()); print("   exc", np.round(ae.mean(0), 1).tolist())

**Result: (to be written)**

## Summary

(to be written)